In [2]:
import os

os.chdir("..")
import torch as th

from reporting.config_loader import load_config, wandb_init_kwargs
from t2Interp.concept_search import KSteer
from t2Interp.mapper import MLPMapper
from t2Interp.T2I import T2IModel
from utils.output_manager import OutputManager
from utils.runningstats import WandbUpdater
from utils.training import Training, TrainingSpec
from utils.utils import preprocess_image

/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `PYTORCH_TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model = T2IModel("CompVis/stable-diffusion-v1-4", device="cuda:0", dtype="float16")

2025-10-27 15:26:09.548 | INFO     | t2Interp.T2I:__init__:108 - Enforcing eager attention implementation for attention pattern tracing. The HF default would be to use sdpa if available. To use sdpa, set attn_implementation='sdpa' or None to use the HF default.
Keyword arguments {'attn_implementation': 'eager', 'tokenizer_kwargs': {}, 'trust_remote_code': False} are not expected by StableDiffusionPipeline and will be ignored.
Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  9.75it/s]
You have d

In [3]:
preprocess_fn = lambda x: preprocess_image(x, 512)
race_labels = {
    "East Asian": 0,
    "Indian": 1,
    "Black": 2,
    "White": 3,
    "Middle Eastern": 4,
    "Latino_Hispanic": 5,
    "Southeast Asian": 6,
}
gt_processing_fn = lambda x: th.tensor(race_labels[x], dtype=th.long)

In [4]:
model._wrappers

{'text_encoder_2': blocks:
 0: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 1: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 2: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 3: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 4: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 5: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 6: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 7: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 8: in_ | attn_in | attn_out | WO_in 

### check latent shape

In [4]:
cache = model.run_with_cache(
    [""],
    accessors=[model.unet_2.down_attn_blocks[0].self_attn_out],
    **{"num_inference_steps": 1, "seed": 40},
)
for k, v in cache.items():
    print(k, v.output.shape)

100%|██████████| 1/1 [00:00<00:00,  2.97it/s]

unet_down_block_attn0_self_attn_out torch.Size([2, 4096, 320])


In [ ]:
kwargs = {
    "preprocess_fn": preprocess_fn,
    "gt_processing_fn": gt_processing_fn,
    # "subset" :100,
    "val_split": "val",
    "dataset_column": "image",
    "ground_truth_column": "race",
    "use_val": True,
    "steps": 200,
    "denoising_step": 0,
    "training_device": "cuda:0",
    "data_device": "cpu",
    "autocast_dtype": th.bfloat16,
    "d_submodule": 4096 * 320,
    "log_steps": 5,
    "refresh_batch_size": 64,
    "out_batch_size": 16,
    "use_memmap": True,
    "cache_activations": True,
    "run_name": "training_race_steering_mlp",
}

dataset = "nirmalendu01/fairface-trainval-race-balanced-200"
accessor = model.unet_2.down_attn_blocks[0].self_attn_out
mapper = MLPMapper(input_dim=4096 * 320, hidden_dim=4096, output_dim=7)
loss_fn = th.nn.CrossEntropyLoss()
optimizers = [th.optim.Adam(mapper.parameters(), lr=1e-5)]
steering = KSteer(model)

wandb_config = load_config("reporting/config.yaml")
wandb_config["wandb"].update({"run_name": "training_race_steering_mlp"})
init_kwargs = wandb_init_kwargs(wandb_config)
stats_updaters = [WandbUpdater(init_kwargs=init_kwargs)]

output_manager = OutputManager(**kwargs)
callbacks = [output_manager.write_metadata, output_manager.save_best_ckpt]

model.pipeline.set_progress_bar_config(disable=True)
spec = TrainingSpec(
    name="training_race_steering_mlp",
    fn=steering.fit,
    stats_updaters=stats_updaters,
    callback_fns=callbacks,
    args=(dataset, accessor, mapper, loss_fn, optimizers),
    kwargs=kwargs,
)
Training(spec).run_trainer()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


AssertionError: OutputManager requires a run_name argument

### ksteer - using the learnt classifier

In [ ]:
from itertools import tee

from dictionary_learning.utils import hf_dataset_to_generator

from t2Interp.concept_search import KSteer
from t2Interp.intervention import ReplaceIntervention, run_intervention
from utils.output import Output
from utils.text_image_buffer import t2IActivationBuffer
from utils.utils import BatchIterator

race_labels = {
    "East Asian": "East Asian",
    "Indian": "Indian",
    "Black": "Black",
    "White": "White",
    "Middle Eastern": "Middle Eastern",
    "Latino_Hispanic": "Latino Hispanic",
    "Southeast Asian": "Southeast Asian",
}
preprocess_fn = lambda x: f"A photo of a {race_labels[x]} person."


def get_activations_steer_save(model: T2IModel, dataset, accessor, mapper, **kwargs):
    prompts, buffer = tee(
        BatchIterator(
            hf_dataset_to_generator(dataset, **kwargs), batch_size=kwargs.get("out_batch_size", 1)
        )
    )
    d_sub = kwargs.pop("d_submodule", kwargs.pop("d_submodule", None))
    buffer = t2IActivationBuffer(buffer, model, accessor, d_submodule=d_sub, **kwargs)
    ksteer = KSteer(model)
    generate_kwargs = {}
    if "guidance_scale" in kwargs:
        generate_kwargs["guidance_scale"] = kwargs["guidance_scale"]
    imgs = []
    for ps, activations in zip(iter(prompts), iter(buffer)):
        steered_activation = ksteer.steer(activations, target_idx=[1], mapper=mapper, **kwargs)
        output = run_intervention(
            model,
            ps,
            interventions=[
                ReplaceIntervention(model=model, envoys=[accessor], steering_vec=steered_activation)
            ],
            **kwargs,
        )

        imgs.append(output.preds)
    return Output(preds=imgs)

In [ ]:
kwargs = {
    "preprocess_fn": preprocess_fn,
    "dataset_column": "race",
    "steps": 1,
    "denoising_step": 0,
    "data_device": "cpu",
    "autocast_dtype": th.bfloat16,
    "d_submodule": 4096 * 320,
    "refresh_batch_size": 16,
    "out_batch_size": 4,
    "run_name": "training_race_steering_mlp",
    "num_inference_steps": 50,
    "start_step": 0,
    "end_step": 1,
    "alpha": 1.0,
}

model.pipeline.set_progress_bar_config(disable=True)

dataset = "nirmalendu01/fairface-trainval-race-balanced-200"
accessor = model.unet_2.down_attn_blocks[0].self_attn_out
mapper = MLPMapper(input_dim=4096 * 320, hidden_dim=4096, output_dim=7).to(
    "cuda:0", dtype=th.bfloat16
)
mapper.load_state_dict(
    th.load("./runs/steer_mlp_train_steering_20251021-050122/artifacts/best_ckpt.pt")
)

img = get_activations_steer_save(model, dataset, accessor, mapper, **kwargs)

['A photo of a Latino Hispanic person.', 'A photo of a Southeast Asian person.', 'A photo of a East Asian person.', 'A photo of a Latino Hispanic person.']
0 1
torch.Size([8, 4096, 320]) torch.Size([4, 1310720])


In [23]:

kwargs = {
    "preprocess_fn": preprocess_fn,
    "dataset_column": "race",
    "steps": 1,
    "denoising_step": 0,
    "data_device": "cpu",
    "autocast_dtype": th.bfloat16,
    "d_submodule": 4096 * 320,
    "refresh_batch_size": 16,
    "out_batch_size": 4,
    "run_name": "training_race_steering_mlp",
    "num_inference_steps": 50,
    "start_step": 0,
    "end_step": 1,
    "alpha": 1.0,
}

imgs = {step: [] for step in range(1, 20, 1)}
for step in range(1, 20, 1):
    with model.generate(
        "a photo of a asian person", seed=40, **{**kwargs, "num_inference_steps": step}
    ):
        imgs[step] = model.output.save()

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 23.71it/s]


### activation separation at different unet blocks for race

In [4]:
from pathlib import Path
from typing import Any

from utils.text_image_buffer import TextImageActivationBuffer

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch as th
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# --- your pieces assumed available in scope ---
# - hf_dataset_to_generator(dataset, **kwargs)
# - TextImageActivationBuffer(generator, model, accessor, d_submodule=..., **kwargs)
# - BatchIterator(gen, batch_size)
# - model, accessors, kwargs (from your message)


def _ensure_dir(p: str | Path):
    Path(p).mkdir(parents=True, exist_ok=True)


def _tsne_plot(
    X: np.ndarray,
    y: np.ndarray,
    title: str,
    out_path: str,
    pca_components: int | None = 50,
    tsne_perplexity: int | None = None,
    tsne_iter: int = 1000,
    random_state: int = 42,
):
    """
    X: (N, D), y: (N,)
    """
    assert len(X) == len(y), "X and y length mismatch"
    if len(X) < 10:
        print(f"[warn] Too few points ({len(X)}) to plot t-SNE for {title}. Skipping.")
        return

    X_proc = X
    if pca_components is not None and X.shape[1] > pca_components:
        pca = PCA(n_components=pca_components, random_state=random_state)
        X_proc = pca.fit_transform(X)

    # pick a safe perplexity
    if tsne_perplexity is None:
        # rule of thumb: < N/3 and at least 5
        tsne_perplexity = max(5, min(30, (len(X_proc) // 3) - 1))

    tsne = TSNE(
        n_components=2,
        perplexity=tsne_perplexity,
        # n_iter=tsne_iter,
        learning_rate="auto",
        init="pca",
        random_state=random_state,
        verbose=1,
    )
    X2 = tsne.fit_transform(X_proc)

    # plot
    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(X2[:, 0], X2[:, 1], c=y, s=8, alpha=0.8)
    plt.title(title)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    # class legend via unique labels
    classes = np.unique(y)
    handles = []
    for c in classes:
        handles.append(plt.Line2D([], [], marker="o", linestyle="None", label=str(c)))
    plt.legend(handles=handles, title="race", loc="best", fontsize=8)
    _ensure_dir(os.path.dirname(out_path))
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"[plot] saved {out_path}")


def cache_activations_for_accessor(
    dataset: str,
    accessor,  # must have .attr_name
    model,
    d_submodule: int | None,
    start_step: int,
    end_step: int,
    cache_dir: str | Path,
    max_items: int | None = None,  # cap for quick runs; None = all
    force_refresh: bool = False,  # NEW: ignore existing npz and recompute if True
    **kwargs,
) -> dict[tuple[str, int], tuple[np.ndarray, np.ndarray]]:
    """
    Returns {(accessor_name, denoising_step): (X, y)}.
    Uses cached shards at <cache_dir>/<attr_name>/acts_stepXXX.npz when available,
    otherwise computes and writes them. If max_items is set and a cached shard
    is larger, it will be truncated in-memory (the cached file is left as-is).
    """
    results: dict[tuple[str, int], tuple[np.ndarray, np.ndarray]] = {}

    def log(msg: str):
        print(msg)

    base_dir = Path(cache_dir) / accessor.attr_name
    _ensure_dir(base_dir)

    for step in range(start_step, end_step + 1):
        shard_path = base_dir / f"acts_step{step:03d}.npz"

        # --- Fast path: load cache if it exists and not forcing refresh ---
        if shard_path.exists() and not force_refresh:
            try:
                data = np.load(shard_path)
                X = data["X"]
                y = data["y"]
                # optional in-memory truncate to respect max_items for this session
                if max_items is not None and len(y) > max_items:
                    X = X[:max_items]
                    y = y[:max_items]
                log(f"[cache] loaded {shard_path}  ->  X={X.shape}, y={y.shape}")
                results[(accessor.attr_name, step)] = (X, y)
                continue
            except Exception as e:
                log(f"[cache] failed to load {shard_path} ({e}); will recompute.")

        # --- Compute path: build generators/buffers and collect ---
        log(f"[{accessor.attr_name}] collecting activations at denoising_step={step}")

        gen_img = hf_dataset_to_generator(dataset, **kwargs)
        gen_lbl = hf_dataset_to_generator(
            dataset,
            **{
                **kwargs,
                "dataset_column": kwargs.get("ground_truth_column", "ground_truth"),
                "preprocess_fn": kwargs.get("gt_processing_fn", None),
            },
        )

        step_kwargs = {**kwargs, "denoising_step": step}
        act_buffer = TextImageActivationBuffer(
            gen_img, model, accessor, d_submodule=d_submodule, **step_kwargs
        )
        lbl_iter = BatchIterator(gen_lbl, kwargs.get("out_batch_size", 1))

        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)

        all_X: list[np.ndarray] = []
        all_y: list[np.ndarray] = []
        n_seen = 0

        for acts, gts in zip(iter(act_buffer), iter(lbl_iter)):
            if isinstance(acts, th.Tensor):
                x_np = acts.detach().float().cpu().numpy()
            elif isinstance(acts, np.ndarray):
                x_np = acts
            else:
                x_np = np.asarray(acts)

            if isinstance(gts, th.Tensor):
                y_np = gts.detach().long().cpu().numpy()
            elif isinstance(gts, np.ndarray):
                y_np = gts.astype(np.int64)
            else:
                y_np = np.asarray(gts, dtype=np.int64)

            y_np = y_np.reshape(-1)

            all_X.append(x_np)
            all_y.append(y_np)

            n_seen += len(y_np)
            if max_items is not None and n_seen >= max_items:
                break

        if not all_X:
            log(f"[warn] no activations gathered for step={step} on {accessor.attr_name}")
            continue

        X = np.concatenate(all_X, axis=0)
        y = np.concatenate(all_y, axis=0)

        if max_items is not None and len(y) > max_items:
            X = X[:max_items]
            y = y[:max_items]

        # Write/overwrite cache on disk
        try:
            np.savez_compressed(shard_path, X=X, y=y)
            log(f"[cache] wrote {shard_path}  ->  X={X.shape}, y={y.shape}")
        except Exception as e:
            log(f"[cache] FAILED writing {shard_path}: {e}")

        results[(accessor.attr_name, step)] = (X, y)

    return results


def run_cache_and_tsne(
    dataset: str,
    accessors: list[Any],
    model,
    kwargs: dict[str, Any],
    d_submodule: int | None,
    out_dir: str | Path = "analysis/tsne",
    max_items_per_combo: int | None = 8000,  # adjust for speed/VRAM
    pca_components: int | None = 50,
    tsne_perplexity: int | None = None,
    tsne_iter: int = 1000,
):
    _ensure_dir(out_dir)

    start_step = kwargs.pop("start_step", 0)
    end_step = kwargs.pop("end_step", start_step)

    for accessor in accessors:
        # 1) cache (and collect in-memory) for this accessor
        results = cache_activations_for_accessor(
            dataset=dataset,
            accessor=accessor,
            model=model,
            d_submodule=d_submodule,
            start_step=start_step,
            end_step=end_step,
            cache_dir=Path(out_dir) / "cache",
            max_items=max_items_per_combo,
            **kwargs,
        )

        # 2) t-SNE per (accessor, step)
        for (acc_name, step), (X, y) in results.items():
            title = f"{acc_name} | step={step} | N={len(y)} D={X.shape[1]}"
            png = Path(out_dir) / f"tsne__{acc_name.replace('/', '_')}__step{step:03d}.png"
            _tsne_plot(
                X,
                y,
                title=title,
                out_path=str(png),
                pca_components=pca_components,
                tsne_perplexity=tsne_perplexity,
                tsne_iter=tsne_iter,
            )


# -------------------------------
# Example call
# -------------------------------
RACE_LABELS = {
    "East Asian": 0,
    "Indian": 1,
    "Black": 2,
    "White": 3,
    "Middle Eastern": 4,
    "Latino_Hispanic": 5,
    "Southeast Asian": 6,
}


def preprocess_fn(x):
    return preprocess_image(x, 512)


def race_processing_fn(x):
    return th.tensor(RACE_LABELS[x], dtype=th.long)


kwargs = {
    "preprocess_fn": preprocess_fn,
    "dataset_column": "image",
    "steps": 1,
    "denoising_step": 0,
    "data_device": "cpu",
    "autocast_dtype": th.float16,
    "refresh_batch_size": 16,
    "out_batch_size": 4,
    "run_name": "training_race_steering_mlp",
    "num_inference_steps": 50,
    "start_step": 0,
    "end_step": 10,  # <<< sweep first 11 steps as a demo; set as you like
    "alpha": 1.0,
    "ground_truth_column": "race",
    "gt_processing_fn": race_processing_fn,
    "dataset_split": "val",  # your loader should respect this
}

# your given list:
accessors = [
    model.unet_2.down_attn_blocks[0].self_attn_out,
    model.unet_2.down_attn_blocks[0].cross_attn_out,
]
# and your d_sub:
d_sub = 4096 * 320

# Run
run_cache_and_tsne(
    dataset="nirmalendu01/fairface-trainval-race-balanced-200",
    accessors=accessors,
    model=model,
    kwargs=kwargs,
    d_submodule=d_sub,
    out_dir="analysis/tsne_fairface_race",
    max_items_per_combo=8000,  # reduce to e.g. 2000 for quick tests
    pca_components=50,
    tsne_perplexity=None,  # auto
    tsne_iter=1000,
)

[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step000.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step001.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step002.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step003.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step004.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step005.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache/unet_down_block_attn0_self_attn_out/acts_step006.npz  ->  X=(1388, 1310720), y=(1388,)
[cache] loaded analysis/tsne_fairface_race/cache

100%|██████████| 50/50 [00:09<00:00,  5.46it/s]
